# Nurse Rostering 

## I. Khái niệm

- Bài toán sắp xếp điều dưỡng - $NRP$ có mục tiêu tạo ra một lịch làm việc khả thi cho $n$ điều dưỡng trong một giai đoạn kế hoạch gồm $d$ ngày bằng `SAT solver`

- $X_{i,j,k}$ là nhân viên $i$ trong ngày $j$ làm việc ca k (hoặc không làm ca nào)
    - $1 \le i \le n$: Vì có $n$ nhân viên
    - $1 \le j \le d$: Vì có $d$ ngày
    - $k \in \{D, E, N, O\}$ : $D$ là ca ngày, $E$ là ca tối, $N$ là ca đêm, $O$ là không làm ca nào trong ngày

## II. Cách chuyển ALK thành AMK

1. Ý tưởng:
    - Ít nhất $k$ tương đương với nhiều nhất $w - k$, "Tối đa $k$ biến trong số $w$ biến mang giá trị $True$" tương đương với ràng buộc "Ít nhất $w−k$ biến trong số $w$ biến đó mang giá trị $False$"

$$ \bigwedge_{i=0}^{n-w}\left(\sum_{j=i+1}^{i+w} X_j \ge k\right) \equiv \bigwedge_{i=0}^{n-w}\left(\sum_{j=i+1}^{i+w} \neg X_j \le w - k\right)\qquad 1 \le k < w$$

## III.Các ràng buộc

1. **Mỗi ngày** mỗi điều dưỡng làm **tối đa 1 ca làm việc**
    - Dạng triển khai:
        - Ngày 1 người 1 làm tối đa 1 ca: $X_{1,1,D} + X_{1,1,E} + X_{1,1,N} + X_{1,1,O} = 1$ \
          $...$
    - Dạng tổng quát:
      $$ \bigwedge_{i = 1} ^ n \bigwedge_{j = 1}^d \sum_{k \in \{D, E, N, O\}} X_{i, j, k} = 1$$
> thực tế nếu không có điều kiện (1) phải có điều kiện nếu $X_{i,j,O} = 1 \leftrightarrow X_{i,j,D} + X_{i, j, E} + X_{i, j, N} = 0$. Tuy nhiên điều kiện (1) đã bao gồm điều kiện trên. Và nếu bỏ điều kiện (1) này bài toán sẽ **phức tạp hơn nhiều**

2. Trong mỗi **7 ngày liên tiếp**, mỗi nhân viên làm **tối đa 6 ca làm việc**
   - Dạng triển khai (minh họa của người thứ 1)\
$X_{1,1,O} + X_{1,2,O} + X_{1, 3, O} + X_{1,4,O} + X_{1,5,O} + X_{1, 6, O} + X_{1,7,O} \ge 1$ \
$ \ \ \ \\ \ \\ \ \ \ \ \  \  \ \  \ \ \\ \ X_{1,2,O} + X_{1,3,O} + X_{1, 4, O} + X_{1,5,O} + X_{1,6,O} + X_{1, 7, O} + X_{1,8,O} \ge 1$ \
$\dots$

   - Dạng tổng quát \
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 6} \sum_{j'= j} ^ {j + 6} \neg X_{i, j', O} \le 6 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 6} \sum_{j'= j} ^ {j + 6}  X_{i, j', O} \ge 1 )$$
    > - Phần trong dấu ngoặc chính là các dàng buộc dạng `cầu thang` (ladder), phép $\land$ ở ngoài có là bởi vì có $n$ nhân viên khác nhau
    > - Tương đương: **phủ định** của **ngày nghỉ** là **ngày làm việc**

3. Trong **14 ngày liên tiếp**, phải có **ít nhất 4 ngày nghỉ**
- Dạng triển khai\
$X_{1,1,O} + X_{1,2,O} + X_{1, 3, O} + X_{1,4,O} + X_{1,5,O} + X_{1, 6, O} + X_{1,7,O} + X_{1,8,O} + X_{1,9,O} + X_{1, 10, O} + X_{1,11,O} + X_{1,12,O} + X_{1, 13, O} + X_{1,14,O}\le 4$ \
$ \ \ \ \\ \ \\ \ \ \ \ \  \  \ \  \ \ \\ \ X_{1,2,O} + X_{1,3,O} + X_{1, 4, O} + X_{1,5,O} + X_{1,6,O} + X_{1, 7, O} + X_{1,8,O} + X_{1,9,O} + X_{1, 10, O} + X_{1,11,O} + X_{1,12,O} + X_{1, 13, O} + X_{1,14,O} + X_{1,15,O} \le 4$ \
$\dots$
- Dạng tổng quát \
$$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', O} \ge 4 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  \neg X_{i, j', O} \le 10) $$

4. Trong **14 ngày liên tiếp**, phải có **ít nhất 4 ca tối**
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', N} \ge 4 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13} \neg X_{i, j', N} \le 10)$$

5. Trong **14 ngày liên tiếp**, có **tối đa 8 ca tối**
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', E} \le 8 )$$

6. Trong mỗi **28 ngày liên tiếp**, phải có **ít nhất 20 ca làm việc**
   - Tương đương: **phủ định** của **ngày nghỉ** là **ngày làm việc**  \
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 27} \sum_{j'= j} ^ {j + 27}  \neg X_{i, j', O} \ge 20 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 27} \sum_{j'= j} ^ {j + 27}   X_{i, j', O} \le 8 )$$

7. Trong mỗi **14 ngày liên tiếp**, có **tối đa 4 ca đêm**
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', N} \le 4 )$$

8. Trong mỗi **14 ngày liên tiếp**, phải có **ít nhất 1 ca đêm**
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', N} \ge 1 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  \neg X_{i, j', N} \le 13 )$$

9. Trong **7 ngày liên tiếp**, phải có **ít nhất 2 ca tối hoặc ca đêm**

- Dạng triển khai \
$X_{1,1,E} + X_{1,1,N} + X_{1, 2, E} + X_{1,2,N} + X_{1,3,E} + X_{1, 3, N} + \dots + X_{1, 14, E} + X_{1,14,N}\le 4$ \
$\ \ \ \ \ \ \ \ \ \ \ \  \ \ \ \underline{X_{1, 2, E} + X_{1,2,N} + X_{1,3,E} + X_{1, 3, N}   + \dots + X_{1, 14, E} + X_{1,14, N} + X_{1,15,E} + X_{1,15,N} \le 4}$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ X_{1, 2, E} + X_{1,2,N} +  \dots + X_{1, 14, E} + X_{1,14,N} + X_{1,15,E} + X_{1,15,N} + X_{1,16,E} + X_{1,16,N} + \le 4 $ \
**suy ra** \
$X_{1,1,E} + X_{1,1,N} + X_{1, 2, E} + X_{1,2,N} + X_{1,3,E} + X_{1, 3, N} + \dots + X_{1, 14, O} + X_{1,14,O}\le 4$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ X_{1, 2, E} + X_{1,2,N} + X_{1,3,E} + X_{1, 3, N} + \dots + X_{1, 13, O} + X_{1,14,O} + X_{1,15,E} + X_{1,15,N} + \le 4 $ 
- Dạng tổng quát \
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 6} \sum_{j'= j} ^ {j + 6}  X_{i, j', E} + X_{i, j', N} \ge 2 ) \leftarrow  \bigwedge_{i = 1}^n (\bigwedge_{m = 1}^ {2d - 11} \sum_{j'= m} ^ {j + 11}  Y_{i, z} \ge 2 )  \equiv  \bigwedge_{i = 1}^n (\bigwedge_{m = 1}^ {2d - 11} \sum_{j'= m} ^ {j + 11}  \neg Y_{i, z} \le 5 )$$
  
   Trong đó:
$
Y_{i,z} =
\begin{cases}
x_{i,j,E} & \text{khi } z = 3(j - 1) + 1 \\
x_{i,j,N} & \text{khi } z = 3(j - 1) + 2
\end{cases}
$
\
\
(*Đọc phần mở rộng để biết ý tưởng*)

10. Trong mỗi **7 ngày liên tiếp**, có **tối đa 4 ca tối hoặc ca đêm**
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 6} \sum_{j'= j} ^ {j + 6}  X_{i, j', E} + X_{i, j', N} \le 4 ) \leftarrow \bigwedge_{i = 1}^n (\bigwedge_{m = 1}^ {2d - 11} \sum_{j'= m} ^ {j + 11}  Y_{i, z} \le 4 ) $$
   Trong đó:
$
Y_{i,z} =
\begin{cases}
x_{i,j,E} & \text{khi } z = 3(j - 1) + 1 \\
x_{i,j,N} & \text{khi } z = 3(j - 1) + 2
\end{cases}
$
\
\
(*Đọc phần mở rộng để biết ý tưởng*)

11. **Không được phép làm ca đêm vào các ngày liên tiếp**
    $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 1} \sum_{j'= j} ^ {j + 1}  X_{i, j', N} \le 1 )  $$

## IV. Code

In [1]:
from pysat.solvers import Glucose3

from encoding.ladder_encoder import LadderEncoder
from encoding.sinz_encoder import SinzCardinalityEncoder

In [2]:
class NurseRostering:
    def __init__(self, solver, day, n_nurse):
        self.next_id = 1
        self.d = day
        self.n = n_nurse
        self.shift = ['D', 'E', 'N', 'O']
        self.var_map = {}
        self.g = solver

        for nurse in range(self.n):
            for day in range(self.d):
                for shift in self.shift:

                    key = (nurse, day, shift)

                    self.var_map[key] = self.next_id
                    self.next_id += 1

    def var(self, nurse: int, day: int, shift: str) -> int:
        return self.var_map[(nurse, day, shift)]

    def encoding(self):
        self.constraint_1()
        self.constraint_2()
        self.constraint_3()
        self.constraint_4()
        self.constraint_5()
        self.constraint_6()
        self.constraint_7()
        self.constraint_8()
        self.constraint_9()
        self.constraint_10()
        self.constraint_11()
        #self.constraint_fixed_schedule()
       


    # Mỗi ngày mỗi điều dưỡng làm tối đa 1 ca làm việc
    def constraint_1(self):
        for i in range(self.n):
            for j in range(self.d):
                curr_var = [self.var(i, j, k) for k in self.shift]
    
                self.g.add_clause(curr_var)
                for a in range(len(curr_var)):
                    for b in range(a + 1, len(curr_var)):
                        self.g.add_clause([-curr_var[a], -curr_var[b]])
                    
    # Trong mỗi 7 ngày liên tiếp, mỗi nhân viên làm tối đa 6 ca làm việc
    def constraint_2(self):
        for i in range (self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'O') for j in range (self.d)]
            self.next_id = encoder.ladder_alk(self.g, curr_var, 1, 7)
            
    # Trong 14 ngày liên tiếp, phải có ít nhất 4 ngày nghỉ
    def constraint_3(self):
        for i in range (self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'O') for j in range (self.d)]
            self.next_id = encoder.ladder_alk(self.g, curr_var, 4, 14)
    
    # Trong 14 ngày liên tiếp, phải có ít nhất 4 ca tối
    def constraint_4(self):
        for i in range(self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'E') for j in range(self.d)]
            self.next_id = encoder.ladder_alk(self.g, curr_var, 4, 14)
            
    # Trong 14 ngày liên tiếp, có tối đa 8 ca tối
    def constraint_5(self):
        for i in range(self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'E') for j in range(self.d)]
            self.next_id = encoder.ladder_amk(self.g, curr_var, 8, 14)
    
    # Trong mỗi 28 ngày liên tiếp, phải có ít nhất 20 ca làm việc
    def constraint_6(self):
        for i in range(self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'O') for j in range(self.d)]
            self.next_id = encoder.ladder_amk(self.g, curr_var, 8, 28)
            
    # Trong mỗi 14 ngày liên tiếp, có tối đa 4 ca đêm
    def constraint_7(self):
        for i in range(self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'N') for j in range(self.d)]
            self.next_id = encoder.ladder_exk(self.g, curr_var, 4, 14)

    # Trong mỗi 14 ngày liên tiếp, phải có ít nhất 1 ca đêm
    def constraint_8(self):
        for i in range(self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'N') for j in range(self.d)]
            self.next_id = encoder.ladder_alk(self.g, curr_var, 1, 14)
            
    # Trong 7 ngày liên tiếp, phải có ít nhất 2 ca tối hoặc ca đêm
    def constraint_9(self):
        for i in range(self.n):
            encoder = LadderEncoder(self.next_id)
            flattened_Y = []
            for j in range(self.d):
                v_E = self.var(i, j, 'E')
                v_N = self.var(i, j, 'N')
                flattened_Y.append(v_E)
                flattened_Y.append(v_N)
            self.next_id = encoder.ladder_alk(self.g, flattened_Y, 2, 14)
            
    # Trong mỗi 7 ngày liên tiếp, có tối đa 4 ca tối hoặc ca đêm
    def constraint_10(self):
        for i in range(self.n):
            encoder = LadderEncoder(self.next_id) 
            flattened_Y = []
            for j in range(self.d):
                v_E = self.var(i, j, 'E')
                v_N = self.var(i, j, 'N')
                flattened_Y.append(v_E)
                flattened_Y.append(v_N)
            self.next_id = encoder.ladder_amk(self.g, flattened_Y, 4, 14)
            
    # Không được phép làm ca đêm vào các ngày liên tiếp
    def constraint_11(self):
        for i in range (self.n):
            encoder = LadderEncoder(self.next_id) 
            curr_var = [self.var(i, j, 'N') for j in range (self.d)]
            self.next_id = encoder.ladder_amk(self.g, curr_var, 1, 2)

    def constraint_fixed_schedule(self):
        for j in range (1, 14):
            self.g.add_clause([self.var(0, j, 'E')])
            
    ##########################################################################
    # DISPLAY
    ##########################################################################
    
    def display_schedule(self, model: list):
    
        if not model:
            print("Không tìm thấy lịch hợp lệ!")
            return
    
        true_vars = set(model)
    
        nurse_width = 10
        cell_width = 4
    
        total_width = nurse_width + (self.d * (cell_width + 1)) + 1
    
        print("\n" + "=" * total_width)
        print("LỊCH TRỰC ĐIỀU DƯỠNG".center(total_width))
        print("=" * total_width)
    
        header = f"{'Y tá':<{nurse_width}}|"
    
        for j in range(self.d):
            header += f"{j:^{cell_width}}|"
    
        print(header)
        print("-" * total_width)
    
        for i in range(self.n):
    
            row = f"{f'{i:02d}':<{nurse_width}}|"
    
            for j in range(self.d):
    
                assigned_shift = "--"
    
                for shift in self.shift:
    
                    var_id = self.var(i, j, shift)
    
                    if var_id in true_vars:
                        assigned_shift = shift
                        break
    
                row += f"{assigned_shift:^{cell_width}}|"
    
            print(row)
    
        print("=" * total_width)
    
    

In [3]:
if __name__ == "__main__":
    solver = Glucose3()
    
    # Giả sử test với 3 y tá, xếp lịch trong 10 ngày
    num_days = 31
    num_nurses = 1
    
    print(f"Đang khởi tạo bài toán: {num_nurses} y tá, {num_days} ngày...")
    rostering = NurseRostering(solver, day=num_days, n_nurse=num_nurses)
    
    rostering.encoding()
    
    print(f"Tổng số biến đã sinh ra: {rostering.next_id - 1}")
    print("Đang tiến hành giải bài toán...")
    
    is_solved = solver.solve()
    
    if is_solved:
        print("Giải thành công! Đang lấy kết quả...")
        sat_model = solver.get_model()
        rostering.display_schedule(sat_model)
        
    else:
        print("Không tìm được phương án xếp lịch thỏa mãn tất cả ràng buộc (UNSAT)!")

Đang khởi tạo bài toán: 1 y tá, 31 ngày...
Tổng số biến đã sinh ra: 5754
Đang tiến hành giải bài toán...
Giải thành công! Đang lấy kết quả...

                                                                         LỊCH TRỰC ĐIỀU DƯỠNG                                                                         
Y tá      | 0  | 1  | 2  | 3  | 4  | 5  | 6  | 7  | 8  | 9  | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 | 18 | 19 | 20 | 21 | 22 | 23 | 24 | 25 | 26 | 27 | 28 | 29 | 30 |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------
00        | D  | N  | E  | D  | O  | N  | E  | O  | N  | E  | O  | O  | N  | E  | D  | N  | E  | O  | O  | N  | E  | D  | N  | E  | O  | O  | N  | E  | D  | N  | E  |


## V. Mở rộng

**Đặt vấn đề**  
1. Nếu có thể làm tối đa số ca trong 1 ngày
    - Bản chất nếu không có công thức 1 thì bài toán này không thể làm theo cách trên. Vì có điều kiện 1 nên mỗi ngày sẽ chỉ có 2 trạng thái là `làm 1 ca` hoặc không làm ca nào nên bài toán đơn giản
    - Nhưng nếu không có điều kiện 1, một ngày sẽ có thể làm nhiều ca nên không thể dùng tham số $O$ như một phương pháp để giúp bài toán ngắn ngọn hơn 
2. Nếu chỉ được làm giới hạn số ca trong 1 ngày
    - Lúc này ta vẫn làm như cách có thể làm tối đa 1 này nhưng ta phải thêm các dàng buộc AMK cho từng ngày 

**Viết lại các ràng buộc có tham số $O$**

1. Trong mỗi **7 ngày liên tiếp**, mỗi nhân viên làm **tối đa 6 ca làm việc** 
   $$\bigwedge_{i = 1} ^ n \bigwedge_{j = 1} ^ {d - 6} \sum_{j' = j} ^ {j + 6} \sum_{k \in \{D, E, N \}} X_{i, j', k} \le 6$$
   

$(X_{1,1,D} + X_{1,1,E} + X_{1, 1, N}) + (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})  \le 6$ 
$\ \ \\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \  (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + (X_{1,8,D} + X_{1,8,E} + X_{1,8,N})  \le 6$ 
$ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ (X_{1,3,D} + X_{1,3,E} + X_{1,3,N}) + (X_{1,4,D} + X_{1,4,E} + X_{1,4,N}) + \dots + (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})+ (X_{1,8,D} + X_{1,8,E} + X_{1,8,N}) + (X_{1,9,D} + X_{1,9,E} + X_{1,9,N}) \le 6$
$$\dots$$ 


> - Với những điều kiện dạng hình cầu thang nghiêng hơn "mặc định" như thế này thì ta vẫn encoding như sequence constraint như bình thường. Bằng cách nổ xung các biểu thức trung gian, tuy nhiên các biểu thức trung gian không thể chắc chắn $\le k$, nhưng nếu ép nó $\le k$ thì bài làm vẫn đúng *(để ý chiều mũi tên của công thức)
> - Nếu muốn đúng tuyệt đối thì trong quá trình encoding ta sẽ không nối các biểu thức trung gian vì ta không chắc chắn nó $\le k$

$(X_{1,1,D} + X_{1,1,E} + X_{1, 1, N}) + (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})  \le 6$ 
$ \ \ \ \\ \ \\ \ \ \ \ \ \ \ \  \ \  \ \ \\ \ \underline{X_{1,1,E} + X_{1, 1, N}+ (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + X_{1,8,D}  }$ 
$ \ \ \ \\ \ \\ \ \ \ \ \ \ \  \ \ \ \ \ \ \  \ \ \ \ \ \ \ \ \ \ \ \ \ \ \underline{X_{1, 1, N}+ (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + X_{1,8,D} + X_{1,8,E} } $ 
$\  \ \ \ \ \ \ \ \ \ \\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \  (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + (X_{1,8,D} + X_{1,8,E} + X_{1,8,N})  \le 6$ 
$$\dots$$ 
$ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ (X_{1,3,D} + X_{1,3,E} + X_{1,3,N}) + (X_{1,4,D} + X_{1,4,E} + X_{1,4,N}) + \dots + (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})+ (X_{1,8,D} + X_{1,8,E} + X_{1,8,N}) + (X_{1,9,D} + X_{1,9,E} + X_{1,9,N}) \le 6$
$$\dots$$ 



$$\bigwedge_{i = 1} ^ n (\bigwedge_{j = 1} ^ {d - 6} \sum_{j' = j} ^ {j + 6} \sum_{k \in \{D, E, N \}} X_{i, j', k} \le 6) \leftarrow \bigwedge_{i = 1} ^ n (\bigwedge_{m = 1} ^ {3d - 20} \sum_{z = m} ^ {m + 20} Y_{i, z} \le 6)$$ 
Trong đó:
$
Y_{i,z} =
\begin{cases}
x_{i,j,D} & \text{khi } z = 3(j - 1) + 1 \\
x_{i,j,E} & \text{khi } z = 3(j - 1) + 2 \\
x_{i,j,N} & \text{khi } z = 3(j - 1) + 3
\end{cases}
$

> Nếu chỉ được làm $s$ ca một ngày


$$ \bigwedge_{i = 1} ^ n \bigwedge_{j = 1} ^ d \sum_{k \in \{D, E, N\}} X_{i,j,k} \le s$$